In [1]:
# 01 — CLEANING SETUP & INPUT VALIDATION
# OFFLINE ONLY
# ============================================================

from pathlib import Path
import pandas as pd
import re
import html
import unicodedata

ROOT = Path(r"C:\Penelitian Sentimen Bencana Nasional")

INPUT_FILE = (
    ROOT / "01_Raw Data"
    / "outputs" / "reports" / "batch7"
    / "manual_reconciliation"
    / "Batch7_Eligible_Tweets.csv"
)

OUTPUT_DIR = ROOT / "02_Cleaning" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert INPUT_FILE.exists(), (
    f"Input tidak ditemukan: {INPUT_FILE}"
)

df = pd.read_csv(
    INPUT_FILE,
    dtype={"id": "string"},
    keep_default_na=False,
    encoding="utf-8-sig"
)

assert len(df) == 381, (
    f"Expected 381 tweet, ditemukan {len(df)}"
)

assert df["id"].is_unique, "Duplicate tweet ID."
assert df["text"].astype(str).str.strip().ne("").all()
assert df["final_inclusion_decision"].eq("INCLUDE").all()

print("CLEANING INPUT VALIDATION: PASS")
print("Eligible tweets:", len(df))
print("\nDistribution by case:")
display(df["case"].value_counts())


CLEANING INPUT VALIDATION: PASS
Eligible tweets: 381

Distribution by case:


case
sumatra_2025    149
palu_2018       118
lombok_2018     114
Name: count, dtype: int64

In [2]:
# 02 — INITIAL TEXT QUALITY AUDIT
# OFFLINE ONLY | NO TEXT MODIFICATION
# ============================================================

import pandas as pd
import re
import html
import unicodedata

assert len(df) == 381, "Input harus 381 tweet."
assert df["id"].is_unique, "Ada ID duplikat."

audit = df.copy()

# Simpan referensi teks asli tanpa perubahan
audit["text_original"] = audit["text"].astype("string")

# Pemeriksaan dasar
audit["text_length"] = audit["text_original"].str.len()
audit["is_empty"] = (
    audit["text_original"].str.strip().eq("")
)

audit["has_url"] = audit["text_original"].str.contains(
    r"https?://|www\.",
    regex=True,
    case=False,
    na=False
)

audit["has_mention"] = audit["text_original"].str.contains(
    r"(?<!\w)@\w+",
    regex=True,
    na=False
)

audit["has_hashtag"] = audit["text_original"].str.contains(
    r"(?<!\w)#\w+",
    regex=True,
    na=False
)

audit["has_html_entity"] = audit["text_original"].str.contains(
    r"&(?:amp|lt|gt|quot|apos|#\d+|#x[0-9a-f]+);",
    regex=True,
    case=False,
    na=False
)

# Duplikasi exact berdasarkan teks
audit["duplicate_exact_text"] = (
    audit.duplicated(
        subset=["text_original"],
        keep=False
    )
)

# Duplikasi setelah normalisasi ringan:
# Unicode NFKC, HTML unescape, lowercase, whitespace.
# Belum menghapus token ataupun mengubah teks original.
def comparison_key(text):
    text = html.unescape(str(text))
    text = unicodedata.normalize("NFKC", text)
    text = text.casefold()
    text = re.sub(r"\s+", " ", text).strip()
    return text

audit["text_comparison_key"] = (
    audit["text_original"].map(comparison_key)
)

audit["duplicate_normalized_text"] = (
    audit.duplicated(
        subset=["text_comparison_key"],
        keep=False
    )
)

summary = pd.DataFrame([
    ["Total tweets", len(audit)],
    ["Unique tweet IDs", audit["id"].nunique()],
    ["Empty text", int(audit["is_empty"].sum())],
    ["Exact duplicate rows", int(audit["duplicate_exact_text"].sum())],
    ["Normalized duplicate rows", int(audit["duplicate_normalized_text"].sum())],
    ["Contains URL", int(audit["has_url"].sum())],
    ["Contains mention", int(audit["has_mention"].sum())],
    ["Contains hashtag", int(audit["has_hashtag"].sum())],
    ["Contains HTML entity", int(audit["has_html_entity"].sum())],
    ["Minimum text length", int(audit["text_length"].min())],
    ["Maximum text length", int(audit["text_length"].max())],
], columns=["Metric", "Value"])

print("=" * 60)
print("02 — INITIAL TEXT QUALITY AUDIT")
print("=" * 60)
display(summary)

print("\nExact duplicate samples:")
display(
    audit.loc[
        audit["duplicate_exact_text"],
        ["case", "id", "text_original"]
    ].head(15)
)

print("\nNormalized duplicate samples:")
display(
    audit.loc[
        audit["duplicate_normalized_text"],
        ["case", "id", "text_original"]
    ].head(15)
)

print("\nText length statistics:")
display(
    audit.groupby("case")["text_length"]
    .describe()
)

assert not audit["is_empty"].any(), (
    "Ada teks kosong yang perlu diperiksa."
)

print("\nINITIAL TEXT QUALITY AUDIT: PASS")
print("Original text: UNCHANGED")
print("No API call. No data deletion.")


02 — INITIAL TEXT QUALITY AUDIT


,Metric,Value
0,Total tweets,381
1,Unique tweet IDs,381
2,Empty text,0
3,Exact duplicate rows,7
4,Normalized duplicate rows,7
5,Contains URL,232
6,Contains mention,135
7,Contains hashtag,103
8,Contains HTML entity,24
9,Minimum text length,61



Exact duplicate samples:


,case,id,text_original
42,lombok_2018,1028971930978840577,Jika gempa Lombok ditetapkan sebagai bencana n...
43,lombok_2018,1028971930945298433,Jika gempa Lombok ditetapkan sebagai bencana n...
44,lombok_2018,1028968156587540480,Apalagi hingga kini pemerintah pusat belum men...
45,lombok_2018,1028968156130357248,Apalagi hingga kini pemerintah pusat belum men...
74,sumatra_2025,1996195214365757833,Bencana banjir serta longsor melanda sebagian ...
75,sumatra_2025,1996180113290088659,Bencana banjir serta longsor melanda sebagian ...
81,sumatra_2025,1996149775524376777,Bencana banjir serta longsor melanda sebagian ...



Normalized duplicate samples:


,case,id,text_original
42,lombok_2018,1028971930978840577,Jika gempa Lombok ditetapkan sebagai bencana n...
43,lombok_2018,1028971930945298433,Jika gempa Lombok ditetapkan sebagai bencana n...
44,lombok_2018,1028968156587540480,Apalagi hingga kini pemerintah pusat belum men...
45,lombok_2018,1028968156130357248,Apalagi hingga kini pemerintah pusat belum men...
74,sumatra_2025,1996195214365757833,Bencana banjir serta longsor melanda sebagian ...
75,sumatra_2025,1996180113290088659,Bencana banjir serta longsor melanda sebagian ...
81,sumatra_2025,1996149775524376777,Bencana banjir serta longsor melanda sebagian ...



Text length statistics:


,count,mean,std,min,25%,50%,75%,max
case,,,,,,,,
lombok_2018,114.0,164.508772,72.839409,61.0,107.25,142.5,231.5,301.0
palu_2018,118.0,176.567797,76.950587,73.0,98.0,164.0,252.5,362.0
sumatra_2025,149.0,474.550336,752.640509,73.0,197.0,282.0,399.0,6033.0



INITIAL TEXT QUALITY AUDIT: PASS
Original text: UNCHANGED
No API call. No data deletion.
